In [ ]:
library(Seurat)
library(ggplot2)
library(stringr)
library(dplyr)


# 1. Load nod, step1.sctBeforeIntegrate_splitByMouse_nod.rds

In [ ]:
# scdata=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/3.integrated/scdataDF_merged.rds")
# scdata.nod <- subset(scdata, sampleType=="NODSCID")
# scdata.nod[["RNA"]] = split(scdata.nod[["RNA"]], f = scdata.nod$mouse)

# scdata.nod = SCTransform(scdata.nod,vst.flavor = "v2",vars.to.regress =c("percent.mt"),variable.features.n=3000)
# saveRDS(scdata.nod,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step1.sctBeforeIntegrate_splitByMouse_nod.rds")

# 2. integration by harmony

In [ ]:
scdata.nod=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step1.sctBeforeIntegrate_splitByMouse_nod.rds")
scdata.nod = RunPCA(scdata.nod)
scdata.nod = FindNeighbors(scdata.nod, dims = 1:50, reduction = "pca")
scdata.nod = FindClusters(scdata.nod, resolution = 1, cluster.name = "unintegrated_clusters")
scdata.nod = RunUMAP(scdata.nod, dims = 1:50, reduction = "pca", reduction.name = "umap.unintegrated")

scdata.nod = IntegrateLayers(object = scdata.nod, method = HarmonyIntegration, normalization.method = "SCT", assay = "SCT",
                        orig.reduction = "pca", new.reduction = "harmony",#max.iter.cluster = 200L,npcs = 50L,
                        verbose = F)


scdata.nod = RunUMAP(scdata.nod, reduction = "harmony", dims = 1:50, reduction.name = "umap.harmony",verbose = F)
scdata.nod = FindNeighbors(scdata.nod, reduction = "harmony", dims = 1:50,verbose = F)
scdata.nod = FindClusters(scdata.nod, resolution = 0.1,cluster.name = "harmony_cluster",verbose = F)

saveRDS(scdata.nod,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step2.integrate_splitByMouse_nod.rds")


# 3. remove immune cluster and endothelial

In [ ]:
scdata.nod=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step2.integrate_splitByMouse_nod.rds")
scdata.nod

In [ ]:
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata.nod,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=TRUE,label = TRUE,cols=c('0'="#00A087FF",'1'="#4DBBD5FF",'2'="#91D1C2FF",'3'="#3C5488FF",'4'="#F39B7FFF",'5'="#8491B4FF",
  '6'="#fb8072",'7'="#DC0000FF",'8'="#7E6148FF",'9'="#B09C85FF",'10'="#AB82FF",'11'="#00CD00",'12'="#FF6A6A"))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10) 
VlnPlot(scdata.nod,features=c('Epcam','Ptprc','Pecam1','SMALT-Target','Col1a1','Col1a2'),ncol=3)

In [ ]:
scdata.nod@meta.data[1:2,]

In [ ]:
table(scdata.nod$harmony_cluster)
immune <- subset(scdata.nod,subset = harmony_cluster ==7)
immune@meta.data["harmony_cluster"]=NULL
immune = DietSeurat(immune)
saveRDS(immune,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step3.integrate_splitByMouse_nod_immune.rds")
immune

In [ ]:
scdata <- subset(scdata.nod,subset = harmony_cluster !=7)
table(scdata$harmony_cluster)
scdata@meta.data[1:2,]

In [ ]:
scdata@meta.data["unintegrated_clusters"]=NULL
scdata@meta.data["harmony_cluster"]=NULL
scdata = DietSeurat(scdata)
saveRDS(scdata,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step3.integrate_splitByMouse_nod_rmImmuneEndothelial.rds")

In [ ]:
scdata=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step3.integrate_splitByMouse_nod_rmImmuneEndothelial.rds")
# scdata = DietSeurat(scdata)
scdata[["RNA"]] <- JoinLayers(scdata[["RNA"]])
scdata[["RNA"]] = split(scdata[["RNA"]], f = scdata$mouse)
scdata = SCTransform(scdata,vst.flavor = "v2",vars.to.regress =c("percent.mt", "S.Score", "G2M.Score"),variable.features.n=3000)
saveRDS(scdata,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step4.integrate_splitByMouse_nod_rmImmuneEndothelial_splitByMouse.rds")

# 4. FindNeighbors: dims=1:20; FindClusters: resolution = 0.2; RunUMAP:dims = 1:20

In [ ]:
scdata = FindNeighbors(scdata, reduction = "harmony", dims = 1:20,verbose = T)
scdata = FindClusters(scdata, resolution = 0.2,cluster.name = "harmony_cluster",verbose = F)
scdata = RunUMAP(scdata, reduction = "harmony", dims = 1:20, reduction.name = "umap.harmony",verbose = F)
mycols=c('0'="#00A087FF",'1'="#4DBBD5FF",'2'="#91D1C2FF",'3'="#FF6A6A",'4'="#F39B7FFF",'5'="#8491B4FF",
  '6'="#FF8C00",'7'="#DC0000FF",'8'="#7E6148FF",'9'="#B09C85FF",'10'="#AB82FF",'11'="#00CD00")

options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)

options(repr.plot.width =8, repr.plot.height =4)
scdata@active.ident <- factor(Idents(scdata), levels = rev(levels(Idents(scdata))))
p <- DotPlot(scdata, features = c("Tspan1","S100a14","Krt18","Cldn7","Cdh1","Zeb1","Dpp10","Efna5","Tmem163","Syt1",
                                  "H3c3","H2ac4","H3c2","H2ac10","H1f5","Cdc20","Cenpa","Ccnb1","Cdkn3","Ube2c")) +
     RotatedAxis() +
     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
 labs(title = "", x = "", y = "")
p


x <- scdata@meta.data[,c(1,ncol(scdata@meta.data))]
x$cellID <- row.names(x)
print(length(unique(x$cellID)))
x$cellBC <- sapply(str_split(row.names(x),"_"),"[",3)
x$cellBC <- sapply(str_split(x$cellBC,"-"),"[",1)
rename=read.delim("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/rename.txt",header = T)
x <- merge(x, rename,by="sampleID",all=F)
x <- x[,c(3,4,7,6,1,5,2)]
x$cellID <- gsub("-1$","",x$cellID)
print(length(unique(x$cellID)))
x[1:3,]
write.table(x, file="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step5.scdata_metadata_nod_1_20_0.2.txt",sep="\t", quote=FALSE,row.names=F)


In [ ]:
mycols=c('0'="#FF6A6A",'1'="#00A087FF",'2'="#4DBBD5FF",'3'="#F39B7FFF",
         '4'="#FF8C00",'5'="#00CD00",'6'="#AB82FF",'7'="#DC0000FF",'8'="#7E6148FF")
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)

options(repr.plot.width =6, repr.plot.height =4)
scdata@active.ident <- factor(Idents(scdata), levels = rev(levels(Idents(scdata))))
p <- DotPlot(scdata, features =c("Cdh1","Cldn3","Cldn7","Krt8","Krt18","Cdh2","Dpp6","Dpp10","Dcc","Efna5","Zeb1","Tubb3")) +
     RotatedAxis() +
     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
 labs(title = "", x = "", y = "")
p


In [ ]:
scdata=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step7.scdata_node_w_scores.rds")
scdata

In [ ]:
mycols=c('0'="#FF6A6A",'1'="#00A087FF",'2'="#4DBBD5FF",'3'="#F39B7FFF",
         '4'="#FF8C00",'5'="#00CD00",'6'="#AB82FF",'7'="#DC0000FF",'8'="#7E6148FF")
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)


In [ ]:
scdata = RunUMAP(scdata, reduction = "harmony", dims = 1:50, reduction.name = "umap.harmony",verbose = F)
scdata
mycols=c('0'="#FF6A6A",'1'="#00A087FF",'2'="#4DBBD5FF",'3'="#F39B7FFF",
         '4'="#FF8C00",'5'="#00CD00",'6'="#AB82FF",'7'="#DC0000FF",'8'="#7E6148FF")
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)


In [ ]:
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)


# 5. FindAllMarkers

In [ ]:
scdata <- SetIdent(scdata, value = scdata@meta.data$harmony_cluster)
scdata=PrepSCTFindMarkers(scdata)
markers=FindAllMarkers(scdata,test.use = "wilcox",only.pos = T,logfc.threshold = 0.25, assay = "SCT", verbose = F)
write.table(markers,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/nod_markers.csv",sep = "\t",col.names = T,quote=F)

In [ ]:
library(dplyr)
library(ggplot2)
markers <- read.delim("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/nod_markers.csv",header = T)
markers %>%
    group_by(cluster) %>%
    dplyr::filter(avg_log2FC > 1) %>%
    slice_head(n = 5) %>%
    ungroup() -> top5
scdata@active.ident <- factor(Idents(scdata), levels = rev(levels(Idents(scdata))))
p2 <- DotPlot(scdata, features = top5$gene) +
     RotatedAxis() +
     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
 labs(title = "", x = "", y = "")
p2

In [ ]:
markers %>%
    group_by(cluster) %>%
    dplyr::filter(avg_log2FC > 1) %>%
    arrange(desc(avg_log2FC)) %>%
    slice_head(n = 10) %>%
    ungroup() -> top10
write.table(top10, file="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/nod_markers_top10.csv",sep = ",",col.names = T,quote=F)